# Transfer plans

`source.transfer_to(target)` compares the two estimands facet by facet. For every differing
facet it either names the licensing `Assumption` — `unverified` until something checks it —
or returns `blocked` with a reason. No facet passes silently.

| differs | assumption | challenged by | derived from |
|---|---|---|---|
| population | S-admissibility given Z | overlap; moderator interaction | graph (`identify.transport`, Phase 2) |
| window | stationary dynamics; carryover contained | half-life vs window | surface |
| intervention (dose) | surface correct between doses | curvature (chord vs marginal) | surface |
| intervention (version) | version-irrelevance | — | asserted |
| level (unit) | linear aggregation | Jensen gap | surface |
| level (interference) | **blocked** without a declared model | — | — |
| outcome, same dimension | commensurability | not falsifiable | asserted |
| conditioning, collapsible | target strata weights known | strata provenance | population |
| conditioning, non-collapsible | **blocked** | — | — |
| quantity, outcome dimension | **blocked** | — | — |

The status vocabulary is `Verdict`'s: `identified | downgraded | blocked`. Transport *is* an
identification problem.

In [ ]:
from axiom.core import D, Intervention, Outcome, Population, Spec, TimeWindow, Treatment
from axiom.estimands import Estimand, FacetDiff, Level, Quantity, TransferPlan

fertilizer = Treatment(name="fertilizer", dimension=D.currency, unit="USD")
base = dict(
    quantity=Quantity(kind="contrast"),
    treatment=fertilizer,
    intervention=Intervention(doses={"fertilizer": 100.0}, version="granular"),
    reference=Intervention(doses={"fertilizer": 0.0}, version="granular"),
    outcome=Outcome(name="yield_total", dimension=D.outcome, unit="kg"),
    population=Population(name="north", strata={"soil": {"clay": 0.3, "loam": 0.7}}),
    window=TimeWindow(start=0, stop=8),
    level=Level(unit="cluster"),
    dimension=D.outcome,
)
experiment = Estimand(name="experiment_lift", **base)

## Identical facets: identified

The trivial case, but note what it means: two producers that declared the same eight facets
are measuring the same thing, and nothing needs to be assumed.

In [ ]:
plan = experiment.transfer_to(Estimand(name="surface_lift", **base))
print(plan.status, plan.licensed, plan.differing, len(plan.ledger_lines))

## One facet differs: downgraded, with the assumption named

The experiment ran in the north; the decision is about the whole country.

In [ ]:
national = Estimand(name="national_lift", **{**base, "population": Population(name="all", strata={"soil": {"clay": 0.5, "loam": 0.5}})})
plan: TransferPlan = experiment.transfer_to(national)
print(plan.status, plan.differing)
for a in plan.assumptions:
    print(f"  [{a.facet}] {a.name} ({a.state}): {a.statement}")
    print(f"      challenged by: {a.challenged_by}  {a.detail}")

The `s_admissibility` assumption is `unverified`: the selection-diagram verdict that can check
it lives in `identify.transport` (Phase 2) and the resolution in `calibrate.transfer`
(Phase 6). The plan is structural here, and it says so rather than pretending.

## Every differing facet is one ledger line

A `TransferPlan` carries one `LedgerLine` per differing facet, typed, with the content hashes
of both estimands. That is the assumption ledger as a typed diff rather than prose.

In [ ]:
for line in plan.ledger_lines:
    print(line.kind, "|", line.statement, "|", line.detail, "|", line.source[:8], "->", line.target[:8])

## Several facets at once, with corrections

A different dose, a different treatment version, and a longer per-period window. The plan
accumulates assumptions per facet and names the correction operators `calibrate` will need
to apply.

In [ ]:
target = Estimand(name="planning_lift", **{
    **base,
    "intervention": Intervention(doses={"fertilizer": 200.0}, version="liquid"),
    "window": TimeWindow(start=0, stop=12, basis="per_period"),
})
plan = experiment.transfer_to(target)
print(plan.status, plan.differing)
print("corrections:", plan.corrections)
entry: FacetDiff = plan.entry("intervention")
print([a.name for a in entry.assumptions], entry.corrections)

## Blocked facets

A different functional, an outcome of a different dimension, undeclared interference, or
collapsing strata for a non-collapsible quantity (review B1) are not licensed by any assumption
— the plan is `blocked` and the reason says which facet and why.

In [ ]:
cases = {
    "different functional": {"quantity": Quantity(kind="marginal"), "reference": None, "dimension": D.outcome / D.currency},
    "outcome dimension":    {"outcome": Outcome(name="revenue", dimension=D.currency), "dimension": D.currency},
    "interference":         {"level": Level(unit="cluster", interference="within_cluster")},
}
for label, over in cases.items():
    p = experiment.transfer_to(Estimand(name="t", **{**base, **over}))
    print(f"{label:21s} {p.status:8s} {p.reason}")

In [ ]:
ratio_north = Estimand(name="ratio_by_soil", **{**base, "quantity": Quantity(kind="ratio"), "dimension": D.outcome / D.currency, "conditioning": ("soil",)})
ratio_marginal = Estimand(name="ratio_marginal", **{**base, "quantity": Quantity(kind="ratio"), "dimension": D.outcome / D.currency})
print(ratio_north.transfer_to(ratio_marginal).reason)

contrast_by_soil = Estimand(name="lift_by_soil", **{**base, "conditioning": ("soil",)})
p = contrast_by_soil.transfer_to(experiment)     # collapse soil strata; north's soil weights are declared
a = p.entry("conditioning").assumptions[0]
print(p.status, "|", a.name, a.state, "|", p.corrections)

## The plan is a `Spec`

So it serializes into the analysis, hashes, and can be diffed — the ledger is data.

In [ ]:
print(Spec.from_json(plan.to_json()) == plan, plan.content_hash()[:16])